# Aquaculture Model Inference

This notebook demonstrates how to use a trained model for inference on the competition test dataset.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import sys
from pathlib import Path

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from src.trainer import Trainer

# For reproducibility
np.random.seed(42)

# Define paths
DATA_DIR = Path("../data")
EXPERIMENTS_DIR = Path("../experiments")

# Verify directories exist
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

if not EXPERIMENTS_DIR.is_dir():
    raise FileNotFoundError(f"Experiments directory not found: {EXPERIMENTS_DIR.resolve()}")

# Find the latest experiment directory
experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
if not experiment_dirs:
    raise FileNotFoundError("No experiment directories found in ../experiments")

latest_experiment = max(experiment_dirs, key=lambda d: d.stat().st_mtime)
print(f"Using latest experiment: {latest_experiment}")

# Load the trained model
print("Loading trained model...")
trainer = Trainer.load(latest_experiment)
print(f"Model type: {trainer.config.model_type}")

# Load test data
print("Loading test data...")
test_df = pd.read_csv(DATA_DIR / 'Test.csv')
print(f"Test data shape: {test_df.shape}")

# Load sample submission to get format
sample_submission = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
print(f"Sample submission shape: {sample_submission.shape}")

# Prepare test features (same as used during training)
print("Preparing test features...")
# Use the same feature columns as during training
if trainer.feature_names is not None:
    # Use feature names from training
    missing_cols = set(trainer.feature_names) - set(test_df.columns)
    if missing_cols:
        raise ValueError(f"Missing features in test data: {missing_cols}")
    X_test = test_df[trainer.feature_names].values
else:
    # Fallback: use all columns except ID
    test_feature_cols = [col for col in test_df.columns if col not in ['ID']]
    X_test = test_df[test_feature_cols].values

# Make predictions
print("Making predictions...")
predictions = trainer.predict(X_test)

# Get probabilities for ROC AUC calculation (if available)
try:
    probabilities = trainer.predict_proba(X_test)[:, 1]  # Probability of positive class
    has_proba = True
except AttributeError:
    # If predict_proba is not available, use predictions as probabilities
    probabilities = predictions.astype(float)
    has_proba = False
    print("Warning: predict_proba not available, using predictions as probabilities")

# Create submission matching the sample format
print("Creating submission file...")
submission_df = sample_submission.copy()

# Update prediction columns with our model's predictions
# TargetF1: Predicted class labels (for F1 score calculation)
# TargetRAUC: Predicted probabilities (for ROC AUC calculation)
submission_df["TargetF1"] = predictions.astype(int)
submission_df["TargetRAUC"] = probabilities

# Save submission
submission_path = latest_experiment / 'submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")

# Display submission head
print("\nSubmission preview:")
print(submission_df.head())

# Verify submission matches sample format
print(f"\nSubmission shape: {submission_df.shape}")
print(f"Submission columns: {list(submission_df.columns)}")
print("Submission successfully created!")
